<a href="https://colab.research.google.com/github/mdishaq33/chatbot-project/blob/main/chatbot_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installs required packages

In [4]:
!pip install transformers torch

Import Libraries

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [6]:
#Setup Device (GPU/CPU)
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
#Chatbot Program
print("Bot: Hello! I am your AI assistant. Type 'exit' to stop.")

chat_history_ids = None

while True:
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break

    # Convert input to tokens
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Add previous chat history
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids

    # Attention mask (fix warning)
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode output
    bot_output = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

    print("Bot:", bot_output)

Bot: Hello! I am your AI assistant. Type 'exit' to stop.
You: How are you?
Bot: I'm good, how are you?
You: exit
Bot: Goodbye!


In [ ]:
"""
Sample Output
Bot: Hello! I am your AI assistant. Type 'exit' to stop.
You: How are you?
Bot: I'm good, how are you?
You: fine
Bot: That's good.
You: what is NLP?
Bot: Natural Language Processing
You: exit
Bot: Goodbye!
"""